# PathSnap: Colab live demo

[Open this notebook in Google Colab](https://colab.research.google.com/github/vasishtcoffee/ML-HACKATHON/blob/colab-live-demo/notebooks/pathsnap_colab_demo.ipynb)

This notebook checks out the current project branch, uses the committed student data and experiment artifacts, starts the FastAPI service, and lets you choose a student-test trip and scorer. It then displays the live Viterbi-decoded route, map, and evaluation counts.

The demo uses the **student-test** partition only. Evaluator trips are not included. Results are development evidence based on the supplied route alignment; read the notes in `docs/route_decoder.md` before presenting accuracy as ground truth.


In [ ]:
# Clone the exact project revision used by this demo.
from pathlib import Path
import subprocess

REPO = Path('/content/PathSnap')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'colab-live-demo',
        'https://github.com/vasishtcoffee/ML-HACKATHON.git', str(REPO),
    ], check=True)
%cd /content/PathSnap
print('Project checkout:', REPO)


In [ ]:
# Install the project's pinned preprocessing, candidate, model, and API dependencies.
%pip -q install ipywidgets -r requirements/requirements-preprocessing.txt -r requirements/requirements-candidates.txt -r requirements/requirements-baselines.txt -r requirements/requirements-demo.txt


In [ ]:
# Confirm the data and saved experiment artifacts are present in this GitHub revision.
from pathlib import Path
import pandas as pd

required = [
    'data/raw/Student folder.zip',
    'data/processed/observations.csv.gz',
    'data/processed/candidates_student_test.parquet',
    'data/processed/reference_labels_student_test.parquet',
    'experiments/baselines/logistic_regression_student_scores.parquet',
    'experiments/baselines/knn_student_scores.parquet',
    'experiments/random_forest/random_forest_student_scores.parquet',
    'experiments/xgboost/xgboost_student_scores.parquet',
    'experiments/xgboost_tuned/xgboost_tuned_student_scores.parquet',
    'experiments/mlp/mlp_student_scores.parquet',
]
missing = [p for p in required if not Path(p).is_file()]
if missing:
    raise FileNotFoundError('Missing committed demo inputs: ' + ', '.join(missing))
trips = pd.read_csv('data/processed/trip_summary.csv', dtype={'trip_id': str})
print('Committed demo files: OK')
print(trips.groupby('split').trip_id.nunique().to_string())


## Recorded model results

These tables are committed experiment results from the training scripts and give context before the live decode. The API demo below uses the matching saved candidate probabilities on student-test trips. Training every cross-validation fold again is not needed to replay this demo.


In [ ]:
from IPython.display import display

comparison_path = Path('experiments/tree_comparison.csv')
if comparison_path.exists():
    display(pd.read_csv(comparison_path).round(4))
for name in ['baselines', 'random_forest', 'xgboost', 'xgboost_tuned', 'mlp']:
    path = Path('experiments') / name / 'metrics.csv'
    if path.exists():
        metrics = pd.read_csv(path)
        test = metrics.loc[metrics.phase.eq('student_test')]
        if not test.empty:
            print(f'\n{name}: student-test results')
            display(test[['top1_accuracy', 'macro_f1', 'candidate_log_loss']].round(4))


## Start the API

The server accepts requests on localhost inside this Colab runtime. The notebook UI below calls the same `/trips` and `/predict` endpoints used by the browser demo.


In [ ]:
import os
import requests
import subprocess
import sys
import time
from pathlib import Path

API = 'http://127.0.0.1:8000'
api_process = None
try:
    requests.get(API + '/trips', timeout=1).raise_for_status()
    print('An API server is already responding on port 8000.')
except requests.RequestException:
    api_log = open('/content/pathsnap-api.log', 'w')
    api_process = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'pathsnap.api:app', '--host', '127.0.0.1', '--port', '8000'],
        cwd=REPO, stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
    )
    for _ in range(60):
        if api_process.poll() is not None:
            raise RuntimeError(Path('/content/pathsnap-api.log').read_text()[-5000:])
        try:
            response = requests.get(API + '/trips', timeout=1)
            response.raise_for_status()
            break
        except requests.RequestException:
            time.sleep(1)
    else:
        raise TimeoutError('API did not start; inspect /content/pathsnap-api.log')
    print('FastAPI is running; available trip and scorer options:')
print(requests.get(API + '/trips', timeout=10).json())


## Live trip selector

Choose a student-test trip and a scorer, then press **Decode route**. The displayed map and numbers come from a fresh API request in this running Colab session.


In [ ]:
from IPython.display import HTML, display
from html import escape
import ipywidgets as widgets
from google.colab import output
output.enable_custom_widget_manager()

available = requests.get(API + '/trips', timeout=10).json()
trip_picker = widgets.Dropdown(options=available['trip_ids'], description='Trip:', layout=widgets.Layout(width='300px'))
model_picker = widgets.Dropdown(options=list(available['models'].items()), description='Scorer:', layout=widgets.Layout(width='420px'))
run_button = widgets.Button(description='Decode route', button_style='primary')
status = widgets.HTML(value='Choose a trip and scorer, then decode.')
result_box = widgets.Output()

def run_live_decode(_):
    run_button.disabled = True
    status.value = '<b>Decoding…</b>'
    result_box.clear_output(wait=True)
    try:
        response = requests.post(API + '/predict', json={
            'trip_id': trip_picker.value, 'model': model_picker.value,
        }, timeout=180)
        response.raise_for_status()
        result = response.json()
        metrics = result['evaluation']
        status.value = (
            f"<b>{result['scoring']}</b> · trip {result['trip_id']} · "
            f"{result['matched_observations']}/{result['observations']} observations matched · "
            f"{len(result['route_segments'])} route segments · {result['runtime_seconds']} s<br>"
            f"Top-1: {metrics['top1_accuracy']:.1%} · Viterbi: {metrics['viterbi_accuracy']:.1%} · "
            f"{metrics['aligned']} aligned · {metrics['candidate_misses']} candidate misses · "
            f"Excluded: {metrics['ambiguous_excluded']} ambiguous, {metrics['unmatched_excluded']} unmatched"
        )
        with result_box:
            display(HTML(f'<iframe srcdoc="{escape(result["map_html"], quote=True)}" style="width:100%;height:650px;border:0"></iframe>'))
    except Exception as exc:
        status.value = '<b>Request failed:</b> ' + str(exc)
    finally:
        run_button.disabled = False

run_button.on_click(run_live_decode)
display(widgets.VBox([widgets.HBox([trip_picker, model_picker, run_button]), status, result_box]))


## What to explain in the demo

- The scorer supplies a probability for each candidate road arc. Random Forest and XGBoost use the score exports committed by the model experiments.
- Viterbi uses those same per-candidate scores plus directed road connectivity to produce a route sequence.
- The map distinguishes recorded GPS points, nearby road segments, and the decoded route.
- Accuracy is calculated server-side against the student-test reference alignment. That alignment is approximate, so describe the result as a development comparison and include the counts of ambiguous/unmatched labels and candidate misses.
- This notebook replays saved model scores. A final evaluator deployment still needs inference on evaluator-trip candidates, as documented in the README.


In [ ]:
# Stop the API process started by this notebook when the live demo is finished.
if api_process is not None and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)
    print('Stopped the Colab API server.')
else:
    print('No notebook-owned API process to stop.')
